# 🕵️ Dia 04 — Lendo logs de conexão com Zeek e tabelas de sockets

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 4* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-04.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

14h10. A Marta mandou o `conn.log` do sensor Zeek da sede. Nove linhas, nenhum pacote. Três estações do atendimento falaram com `203.0.113.77` de manhã; uma termina em `RSTR`, duas, perto do meio-dia, em `S0`.

Rafa: *"Antes de alguém concluir que o atendimento 'caiu no golpe': me diz, linha por linha, o que cada conexão fez. Em português que a Marta entenda."*

Neste notebook você vai:

1. **Escrever um `zeek-cut` em Python** (colunas pelo nome, hora UTC e BRT).
2. **Contar por estado** com `Counter` (o `sort | uniq -c`).
3. **Decodificar o `history`** letra por letra, em frases.
4. **Jogar o quiz "decifre o history"** (8 perguntas).
5. **Gerar a `day04-timeline-isca.md`** com a leitura em português rascunhada para você revisar.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Funciona sem internet: toda célula que tenta a rede tem plano B embutido.

In [ ]:
# ------------------------------------------------------------
# DADOS DO CASO (Operação Isca — tudo fictício)
# Primeiro tentamos baixar do repositório no GitHub; se não houver
# internet (ou o repo estiver fora do ar), usamos a cópia embutida.
# ------------------------------------------------------------
import urllib.request   # baixa arquivos pela web, sem instalar nada

BASE = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-04/"

# Cópias embutidas: o plano B. São idênticas aos arquivos do repo.
EMBUTIDO = {}
EMBUTIDO['conn.log'] = '#separator \\x09\n#set_separator\t,\n#empty_field\t(empty)\n#unset_field\t-\n#path\tconn\n#open\t2026-10-05-15-10-00\n#fields\tts\tuid\tid.orig_h\tid.orig_p\tid.resp_h\tid.resp_p\tproto\tservice\tduration\torig_bytes\tresp_bytes\tconn_state\thistory\n#types\ttime\tstring\taddr\tport\taddr\tport\tenum\tstring\tinterval\tcount\tcount\tstring\tstring\n1791198842.118000\tCsA1kq3\t10.20.36.21\t53110\t10.20.32.53\t53\tudp\tdns\t0.002\t39\t55\tSF\tDd\n1791198842.131000\tCsA1kq4\t10.20.36.21\t50214\t203.0.113.77\t443\ttcp\tssl\t48.912\t1873\t41250\tSF\tShADadFf\n1791199905.402000\tCsA1kq9\t10.20.36.44\t50388\t203.0.113.77\t443\ttcp\tssl\t61.004\t2290\t41377\tSF\tShADadFf\n1791201737.006000\tCsA1kr2\t10.20.36.44\t50412\t203.0.113.77\t443\ttcp\tssl\t0.611\t517\t0\tRSTR\tShADr\n1791205791.550000\tCsA1kt0\t198.51.100.14\t41022\t10.20.32.10\t3389\ttcp\t-\t0.000\t0\t0\tS0\tS\n1791205791.601000\tCsA1kt1\t198.51.100.14\t41024\t10.20.32.11\t3389\ttcp\t-\t0.000\t0\t0\tS0\tS\n1791211653.200000\tCsA1ku5\t10.20.36.21\t50990\t203.0.113.77\t443\ttcp\t-\t9.012\t0\t0\tS0\tS\n1791211928.900000\tCsA1ku8\t10.20.36.30\t51007\t203.0.113.77\t443\ttcp\t-\t9.008\t0\t0\tS0\tS\n1791212620.000000\tCsA1kv0\t10.20.36.30\t51102\t10.20.32.80\t443\ttcp\tssl\t0.402\t1210\t5320\tSF\tShADadFf\n#close\t2026-10-05-15-10-00\n'
EMBUTIDO['ss-vigia-lab01.txt'] = '# Coletado em 2026-10-05T17:12:40Z na VM de análise da Vigia (vigia-lab01), comando: sudo ss -tunap state established\nNetid Recv-Q Send-Q  Local Address:Port    Peer Address:Port  Process\ntcp   0      0       192.168.56.10:52044   203.0.113.77:443   users:(("firefox-esr",pid=4182,fd=91))\ntcp   0      0       192.168.56.10:52046   203.0.113.77:443   users:(("firefox-esr",pid=4182,fd=97))\ntcp   0      0       192.168.56.10:39810   198.51.100.60:443  users:(("python3",pid=5530,fd=6))\ntcp   0      0       192.168.56.10:22      192.168.56.1:60122 users:(("sshd",pid=1203,fd=4))\n'

def carregar(nome):
    """Devolve o texto do arquivo: da rede se der, senão da cópia embutida."""
    try:
        with urllib.request.urlopen(BASE + nome, timeout=5) as r:   # timeout: não travar a aula
            texto = r.read().decode("utf-8")
        if texto == EMBUTIDO[nome]:                                   # confere com a cópia local
            print(f"🌐 {nome}: baixado do GitHub (idêntico à cópia embutida)")
            return texto
        print(f"⚠️ {nome}: a versão do GitHub difere da embutida; usando a embutida (é a que os hashes do notebook esperam)")
        return EMBUTIDO[nome]
    except Exception as erro:                                         # sem rede, 404, proxy...
        print(f"📦 {nome}: usando a cópia embutida ({type(erro).__name__})")
        return EMBUTIDO[nome]

print(f"{len(EMBUTIDO)} arquivo(s) do caso disponíveis:", ", ".join(EMBUTIDO))

In [ ]:
import hashlib

# Disciplina do Dia 2: hash de tudo o que chega, antes de abrir.
TEXTOS = {nome: carregar(nome) for nome in EMBUTIDO}
for nome, txt in TEXTOS.items():
    print(f"{hashlib.sha256(txt.encode('utf-8')).hexdigest()}  {nome}")

## 1. Um `zeek-cut` em Python

O `conn.log` é texto separado por tabulação. As linhas que começam com `#` são o cabeçalho; a linha `#fields` dá o nome de cada coluna. O `ts` é a hora em **segundos desde 1/1/1970 UTC** (*Unix time*).

In [ ]:
from datetime import datetime, timezone, timedelta

BRT = timezone(timedelta(hours=-3), "BRT")   # horário de Brasília, sem horário de verão

def zeek_cut(texto):
    """Lê um log TSV do Zeek e devolve uma lista de dicionários (coluna → valor)."""
    campos, registros = [], []
    for linha in texto.splitlines():
        if linha.startswith("#fields"):
            campos = linha.split("\t")[1:]                  # nomes das colunas, sem o "#fields"
        elif linha and not linha.startswith("#"):
            reg = dict(zip(campos, linha.split("\t")))      # junta nome com valor
            reg["ts_utc"] = datetime.fromtimestamp(float(reg["ts"]), timezone.utc)   # o -u do zeek-cut
            reg["ts_brt"] = reg["ts_utc"].astimezone(BRT)    # a hora que a Marta entende
            registros.append(reg)
    return registros

CONN = zeek_cut(TEXTOS["conn.log"])
COLS = ["id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p", "proto", "service",
        "duration", "orig_bytes", "resp_bytes", "conn_state", "history"]
for r in CONN:
    print(r["ts_utc"].strftime("%Y-%m-%dT%H:%M:%S%z"), *(r[c] for c in COLS), sep="  ")

## 2. Contar por estado (Passo 3)

`sort | uniq -c | sort -rn` em uma linha de Python: `Counter(...).most_common()`.

In [ ]:
from collections import Counter

for estado, n in Counter(r["conn_state"] for r in CONN).most_common():
    print(f"{n:>4} {estado}")

# E por origem: se uma única origem concentra S0/REJ, é varredura.
falhas = Counter(r["id.orig_h"] for r in CONN if r["conn_state"] in ("S0", "REJ"))
print("\nS0/REJ por origem:", dict(falhas))

## 3. O decodificador de `history`

Maiúsculas = originador, minúsculas = respondente. Cada letra vira um pedaço de frase.

In [ ]:
LETRAS = {
    "s": "SYN", "h": "SYN-ACK", "a": "ACK", "d": "dados", "f": "FIN", "r": "RST",
    "t": "retransmissão", "c": "checksum ruim", "i": "pacote inconsistente", "^": "direção invertida",
}
ESTADOS = {
    "SF": "abriu e fechou normalmente",
    "S0": "SYN sem resposta nenhuma",
    "REJ": "recusada (SYN respondido com RST)",
    "RSTO": "estabelecida e depois resetada pelo ORIGINADOR",
    "RSTR": "estabelecida e depois resetada pelo RESPONDENTE",
    "OTH": "vista pelo meio, sem o começo",
}

def decodificar(history):
    """'ShADr' → 'originador: SYN → respondente: SYN-ACK → ...'"""
    partes = []
    for letra in history:
        quem = "originador" if letra.isupper() else "respondente"   # a caixa da letra diz quem
        partes.append(f"{quem}: {LETRAS.get(letra.lower(), '?')}")
    return " → ".join(partes)

for h in ["ShADadFf", "S", "Sr", "ShADr", "Dd"]:
    print(f"{h:9} = {decodificar(h)}")

## 4. 🎮 Quiz: decifre o history

Para cada `history`, escolha o `conn_state` mais provável: **1** SF · **2** S0 · **3** REJ · **4** RSTO · **5** RSTR. `MODO_JOGO = False` mostra só as respostas.

In [ ]:
OPS = {"1": "SF", "2": "S0", "3": "REJ", "4": "RSTO", "5": "RSTR"}
QUIZ = [
    ("ShADadFf", "SF",   "Conversa completa: abriu, dados nos dois sentidos, FIN de cada lado."),
    ("S",        "S0",   "Só o SYN do originador. Ninguém respondeu."),
    ("Sr",       "REJ",  "SYN e, do respondente, RST: porta fechada."),
    ("ShADr",    "RSTR", "Estabeleceu, o originador mandou dados, o RESPONDENTE resetou (r minúsculo)."),
    ("ShADaR",   "RSTO", "Estabeleceu, trocou ACKs e o ORIGINADOR resetou (R maiúsculo)."),
    ("ShAdDaFf", "SF",   "A ordem das letras muda, o desfecho não: FIN dos dois lados."),
    ("ShADadR",  "RSTO", "Dados nos dois sentidos e o originador abortou com RST."),
    ("ShADadfF", "SF",   "Desta vez o RESPONDENTE fechou primeiro (f antes de F). Continua sendo SF: a leitura diz quem fechou."),
]
MODO_JOGO = True

acertos = 0
for i, (h, certo, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] history = {h}   ({decodificar(h)})")
    if MODO_JOGO:
        r = ""
        while r not in OPS:
            r = input("   conn_state? 1=SF 2=S0 3=REJ 4=RSTO 5=RSTR: ").strip()
        ok = OPS[r] == certo
        acertos += ok
        print("   ✅ Acertou!" if ok else "   ❌ Não foi dessa vez.")
    print(f"   Resposta: {certo} — {porque}")

if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido. 🥷" if acertos >= 6 else "— meta: 6 de 8.")

## 5. A janela dos SMS

Os SMS saíram entre **07h12 e 08h40 BRT**. Quais conexões com o site falso caem dentro dessa janela? (Conversão de fuso feita pelo código, nunca de cabeça.)

In [ ]:
inicio = datetime(2026, 10, 5, 7, 12, tzinfo=BRT)
fim = datetime(2026, 10, 5, 8, 40, tzinfo=BRT)

for r in CONN:
    if r["id.resp_h"] != "203.0.113.77":
        continue                                        # só as conexões com o site falso
    dentro = inicio <= r["ts_utc"] <= fim               # comparar datas com fuso funciona direto
    print(f"{r['ts_brt'].strftime('%H:%M:%S')} BRT  {r['id.orig_h']:12} {r['conn_state']:5} "
          f"{'⬅ durante o disparo' if dentro else ''}")

## 6. 📝 A linha do tempo (Passo 5) com a tabela de sockets

A tabela de sockets da VM da Vigia traz o que o `conn.log` nunca terá: **processo e PID**. As leituras em português abaixo são um **rascunho automático**: revise cada uma. Repare que o rascunho fala de endereços, nunca de pessoas.

In [ ]:
import re
from pathlib import Path

def leitura(r):
    """Rascunho da frase em português claro para uma linha do conn.log."""
    o, d = r["id.orig_h"], f"{r['id.resp_h']}:{r['id.resp_p']}"
    est = r["conn_state"]
    if est == "SF":
        return f"{o} conversou com {d} e encerrou normalmente; enviou {r['orig_bytes']} B, recebeu {r['resp_bytes']} B."
    if est == "S0":
        return f"{o} tentou abrir conexão com {d} e não recebeu resposta (filtrado, fora do ar ou bloqueado no caminho)."
    if est == "REJ":
        return f"{o} tentou {d} e foi recusado: nada escutava na porta."
    if est == "RSTR":
        return f"{o} abriu conexão com {d}, enviou {r['orig_bytes']} B e o servidor encerrou abruptamente sem mandar dados."
    if est == "RSTO":
        return f"{o} abriu conexão com {d} e a abortou com RST."
    return ESTADOS.get(est, "estado não coberto: consulte a doc do Zeek")

linhas = []
for r in CONN:
    linhas.append((r["ts_utc"], r["ts_brt"].strftime("%H:%M:%S"), r["id.orig_h"], f"{r['id.resp_h']}:{r['id.resp_p']}",
                   f"{r['proto']}/{r['service']}", r["duration"], f"{r['orig_bytes']}/{r['resp_bytes']}",
                   r["conn_state"], r["history"], "—", leitura(r)))

# Tabela de sockets: a 1ª linha diz quando foi coletada; as demais, conexões com processo.
ss = TEXTOS["ss-vigia-lab01.txt"].splitlines()
quando = datetime.strptime(re.search(r"(\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ)", ss[0]).group(1),
                           "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=timezone.utc)
for l in ss[2:]:
    c = l.split()                                      # Netid Recv-Q Send-Q Local Peer Process
    proc = re.search(r'"([^"]+)",pid=(\d+)', c[5])     # nome do processo e PID
    linhas.append((quando, quando.astimezone(BRT).strftime("%H:%M:%S"), c[3], c[4], c[0], "ao vivo", "—",
                   "ESTAB", "—", f"{proc.group(1)} (PID {proc.group(2)})",
                   f"Na VM da Vigia, o processo {proc.group(1)} mantinha conexão aberta com {c[4]}."))

linhas.sort(key=lambda x: x[0])                        # ordena por hora UTC
cab = ("| Hora (UTC) | Hora (BRT) | Origem | Destino | Proto/Serviço | Duração | Bytes saída/entrada "
       "| Estado | History | Processo (se conhecido) | Leitura em português claro |")
md_tab = [cab, "|" + "---|" * 11]
for l in linhas:
    md_tab.append("| " + l[0].strftime("%Y-%m-%dT%H:%M:%SZ") + " | " + " | ".join(map(str, l[1:])) + " |")
texto = "\n".join(md_tab) + "\n\n> Rascunho gerado automaticamente. Revisado por: ____ em ____ (UTC).\n"

Path("notas").mkdir(exist_ok=True)
Path("notas/day04-timeline-isca.md").write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download("notas/day04-timeline-isca.md")
except ImportError:
    print("(Fora do Colab: o arquivo ficou na pasta notas/)")

### 🧠 Camada 8 — releia a sua coluna de leitura

Alguma frase diz *"o funcionário"*, *"caiu"*, *"digitou a senha"*? Troque por endereço, porta e hora. O log não sabe quem estava na cadeira nem por quê. O atendimento da sede era justamente quem recebia as reclamações: abrir o link denunciado para entender a queixa é o que um atendente diligente faria.

## ✅ Antes de fechar o Dia 4

- [ ] Hash do `conn.log` e da `ss` na custódia
- [ ] Sabe decodificar `ShADadFf`, `S`, `Sr` e `ShADr` sem consultar
- [ ] Quiz com pelo menos 6/8
- [ ] `notas/day04-timeline-isca.md` revisada linha por linha
- [ ] Fez também a prática original com o Zeek no Docker

**Amanhã (Dia 5):** antes de cada conexão houve uma pergunta ao DNS. O resolvedor da cooperativa guardou quem perguntou por `serraazul-pix.example`, e quando.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>